# CNN Image Classification on CIFAR-10
**Part A of the Unlox Minor Project**

Sections:
1. Imports and data loading
2. Dataset description (shape, classes, sample count)
3. Preprocessing
4. Sample images
5. Build the CNN (baseline) and show the model summary
6. Compile and train
7. Training and validation accuracy / loss
8. Test performance
9. Predictions: actual vs predicted
10. Correct and incorrect predictions
11. Confusion matrix and per-class accuracy
12. *(Optional)* Improved model with regularization

## 1. Imports and data loading

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

## 2. Dataset description
*Report item: dataset description and class names.*
Print everything here and screenshot the output.

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Training images shape :", x_train.shape)   # (50000, 32, 32, 3)
print("Training labels shape :", y_train.shape)   # (50000, 1)
print("Test images shape     :", x_test.shape)    # (10000, 32, 32, 3)
print("Test labels shape     :", y_test.shape)    # (10000, 1)
print()
print("Number of training samples:", x_train.shape[0])
print("Number of test samples    :", x_test.shape[0])
print("Image size                :", x_train.shape[1:], "(height, width, RGB channels)")
print("Number of classes         :", len(class_names))
print("Class names               :", class_names)
print()
print("Pixel value range before normalization:", x_train.min(), "to", x_train.max())

# Images per class (CIFAR-10 is balanced)
print("\nImages per class in training set:")
for i, name in enumerate(class_names):
    print(f"  {i} - {name:<10}: {np.sum(y_train == i)}")

## 3. Preprocessing
* **Images:** divide by 255 so pixel values go from 0-255 to 0-1. This makes training more stable.
* **Labels:** kept as integers (0-9). We use `sparse_categorical_crossentropy`, which accepts integer labels, so no one-hot encoding is needed.

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

print("Pixel value range after normalization:", x_train.min(), "to", x_train.max())
print("Example label:", y_train[0], "->", class_names[y_train[0][0]])

## 4. Sample images

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.show()

## 5. Build the CNN (baseline model)
Structure: Conv -> Pool -> Conv -> Pool -> Conv -> Flatten -> Dense -> Dense(softmax).
Take a screenshot of the `model.summary()` output for the report.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(100, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

model.summary()

## 6. Compile and train
`EarlyStopping` stops training when validation loss stops improving.
`ReduceLROnPlateau` halves the learning rate when validation loss stalls.

`validation_split=0.1` keeps the last 10% of the training images (5,000) aside for validation, so the model trains on 45,000 images. The 10,000 test images are only used for the final evaluation.

**Tip:** in Colab use *Runtime -> Change runtime type -> T4 GPU* before running this cell. Training on CPU takes about 3 minutes per epoch; on GPU it takes seconds.

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=0.001,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_delta=0.001,
    min_lr=1e-5
)

history = model.fit(
    x_train, y_train,
    epochs=10,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr]
)

## 7. Training and validation accuracy and loss
*Report items: training/validation accuracy and training/validation loss.*
If the training line keeps improving while the validation line flattens or rises, the model is overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='training accuracy')
axes[0].plot(history.history['val_accuracy'], label='validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training vs Validation Accuracy')
axes[0].legend(loc='lower right')

# Loss
axes[1].plot(history.history['loss'], label='training loss')
axes[1].plot(history.history['val_loss'], label='validation loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training vs Validation Loss')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 8. Test performance

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}  ({test_acc * 100:.2f}%)")

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal training accuracy  : {final_train_acc * 100:.2f}%")
print(f"Final validation accuracy: {final_val_acc * 100:.2f}%")
print(f"Gap (train - validation) : {(final_train_acc - final_val_acc) * 100:.2f} percentage points")

## 9. Predictions: actual vs predicted
`model.predict` returns 10 probabilities per image (one per class).
`np.argmax` picks the class with the highest probability.

In [ ]:
predictions = model.predict(x_test)                 # shape (10000, 10)
predicted_labels = np.argmax(predictions, axis=1)   # shape (10000,)
actual_labels = y_test.flatten()                    # shape (10000,)

print("predictions shape      :", predictions.shape)
print("First image probabilities:", np.round(predictions[0], 3))
print("First image predicted  :", class_names[predicted_labels[0]])
print("First image actual     :", class_names[actual_labels[0]])

In [ ]:
# Grid of the first 25 test images with actual and predicted labels.
# Green title = correct, red title = wrong.
plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(x_test[i])
    plt.xticks([])
    plt.yticks([])
    colour = 'green' if predicted_labels[i] == actual_labels[i] else 'red'
    plt.title(f"A: {class_names[actual_labels[i]]}\nP: {class_names[predicted_labels[i]]}",
              color=colour, fontsize=9)
plt.suptitle("Sample test images (A = actual, P = predicted)", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Correct and incorrect predictions
*Report item: correct and incorrect predictions.*
We compare predicted labels with actual labels to find which images the model got right and wrong.

In [ ]:
correct_idx = np.where(predicted_labels == actual_labels)[0]
wrong_idx = np.where(predicted_labels != actual_labels)[0]

print("Correct predictions  :", len(correct_idx))
print("Incorrect predictions:", len(wrong_idx))
print(f"Accuracy check       : {len(correct_idx) / len(actual_labels) * 100:.2f}%")

In [ ]:
def show_examples(indices, title, n=10):
    # Show up to n images from the given indices.
    plt.figure(figsize=(14, 6))
    for i, idx in enumerate(indices[:n]):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x_test[idx])
        plt.xticks([])
        plt.yticks([])
        confidence = predictions[idx][predicted_labels[idx]] * 100
        plt.title(f"Actual: {class_names[actual_labels[idx]]}\n"
                  f"Predicted: {class_names[predicted_labels[idx]]}\n"
                  f"Confidence: {confidence:.0f}%", fontsize=9)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_examples(correct_idx, "Correctly classified images")
show_examples(wrong_idx, "Incorrectly classified images")

## 11. Confusion matrix and per-class accuracy
*Used to explain the result in simple words.* Each row is the actual class, each column is the predicted class.
The diagonal shows correct predictions. Large off-diagonal numbers show which classes get confused.

In [ ]:
num_classes = len(class_names)
conf_matrix = tf.math.confusion_matrix(actual_labels, predicted_labels,
                                       num_classes=num_classes).numpy()

plt.figure(figsize=(9, 8))
plt.imshow(conf_matrix, cmap='Blues')
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha='right')
plt.yticks(range(num_classes), class_names)
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.title('Confusion Matrix')

# Write the number in each cell
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, conf_matrix[i, j], ha='center', va='center',
                 color='white' if conf_matrix[i, j] > conf_matrix.max() / 2 else 'black',
                 fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy for each class
per_class_acc = conf_matrix.diagonal() / conf_matrix.sum(axis=1)

print("Per-class accuracy:")
for name, acc in sorted(zip(class_names, per_class_acc), key=lambda x: x[1], reverse=True):
    print(f"  {name:<10}: {acc * 100:.1f}%")

# Most common mistakes: largest off-diagonal entries
off_diag = conf_matrix.copy()
np.fill_diagonal(off_diag, 0)
print("\nTop 5 most common confusions (actual -> predicted):")
for flat_idx in np.argsort(off_diag, axis=None)[::-1][:5]:
    a, p = divmod(flat_idx, num_classes)
    print(f"  {class_names[a]:<10} -> {class_names[p]:<10}: {off_diag[a, p]} times")

### Explain the result in simple words (fill this in for your report)
Use your own numbers from the cells above. A template to adapt:

* The model correctly classified about **___%** of the 10,000 test images.
* Training accuracy reached **___%** but validation accuracy stayed near **___%**. This gap means the model is **overfitting**: it memorizes the training images instead of learning patterns that carry over to new images.
* The best classes were **___** (easy shapes/backgrounds) and the weakest were **___**.
* The most common confusions were **___ vs ___**, because these classes look similar in 32x32 images.

**Challenges:** overfitting, small low-resolution images, slow training on CPU.

**Possible improvements:** data augmentation, dropout, batch normalization, deeper conv blocks, more epochs on GPU, transfer learning.

---
## 12. (Optional) Improved model with regularization
Run this section **after** sections 1-11 so you can compare it with the baseline in your report.
It adds data augmentation, batch normalization, dropout and VGG-style double conv blocks.

**Use a GPU runtime for this section.** Training for up to 50 epochs on CPU takes hours.

In [ ]:
# Keep the baseline results for the before/after comparison
baseline_history = history
baseline_test_acc = test_acc

data_aug = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

improved_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_aug,

    # Block 1
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 3
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # Classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])

improved_model.summary()

In [ ]:
improved_model.compile(optimizer="adam",
                       loss="sparse_categorical_crossentropy",
                       metrics=["accuracy"])

early_stopping2 = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
reduce_lr2 = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)

improved_history = improved_model.fit(
    x_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping2, reduce_lr2]
)

In [ ]:
# Compare baseline and improved model
improved_loss, improved_acc = improved_model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {baseline_test_acc * 100:.2f}%")
print(f"Improved test accuracy: {improved_acc * 100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, title) in zip(axes, [('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    ax.plot(improved_history.history[key], label=f'training {key}')
    ax.plot(improved_history.history['val_' + key], label=f'validation {key}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'Improved model: {title}')
    ax.legend()
plt.tight_layout()
plt.show()

**To get predictions, correct/incorrect examples and the confusion matrix for the improved model:**
set `model = improved_model` in a new cell, then re-run sections 8 to 11.